# Prompt engineering - class demo

# 2a

In [2]:
%pip install python-dotenv --upgrade --quiet langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.6 MB/s eta 0:00:00


In [3]:
from dotenv import load_dotenv
import os
import getpass
!pip install langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

# Low temperature for consistent comparisons
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

Enter your Google API Key: ··········


In [4]:
task = "Write a rejection email to a candidate."

print("--- LAZY PROMPT ---")
print(llm.invoke(task).content)


--- LAZY PROMPT ---
Here are a few options for a rejection email, ranging from a standard template to one for a candidate who interviewed. Choose the one that best fits your situation.

---

**Option 1: Standard Rejection (No Interview)**

This is suitable for candidates who applied but were not selected for an interview.

**Subject: Update on Your Application for [Job Title] at [Company Name]**

Dear [Candidate Name],

Thank you for your interest in the [Job Title] position at [Company Name] and for taking the time to submit your application.

We received a large number of highly qualified applications for this role. While your qualifications are impressive, we have decided to move forward with other candidates whose profiles were a closer match for the specific requirements of this position at this time.

We appreciate you considering [Company Name] as a potential employer and wish you the best of luck in your job search and future endeavors.

Sincerely,

[Your Name]
[Your Title]
[Co

In [5]:
structured_prompt = """
# Context
You are an HR Manager at a quirky startup called 'RocketBoots'.

# Objective
Write a rejection email to a candidate named Bob.

# Constraints
1. Be extremely brief (under 50 words).
2. Do NOT say 'we found someone better'. Say 'the role changed'.
3. Sign off with 'Keep flying'.

# Output Format
Plain text only. No subject line.
"""

print("--- STRUCTURED PROMPT ---")
print(llm.invoke(structured_prompt).content)


--- STRUCTURED PROMPT ---
Hi Bob,

Thanks for your interest in RocketBoots. We appreciate you taking the time.

While your application was strong, the role's requirements have recently changed significantly. We won't be moving forward with your candidacy at this time.

Keep flying,
RocketBoots HR


# 2b


In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.5
)

prompt_zero = "Combine 'Angry' and 'Hungry' into a funny new word."

print("Zero-Shot:")
print(llm.invoke(prompt_zero).content)


Zero-Shot:
The most common and widely recognized funny word for being angry because you're hungry is **Hangry**.


In [7]:
prompt_few = """
Combine words into a funny new word. Give a sarcastic definition.

Input: Breakfast + Lunch
Output: Brunch (An excuse to drink alcohol before noon)

Input: Chill + Relax
Output: Chillax (What annoying people say when you are panicking)

Input: Angry + Hungry
Output:
"""

print("Few-Shot:")
print(llm.invoke(prompt_few).content)


Few-Shot:
Output: Hangry (The highly scientific justification for why someone is being a complete monster until their next meal)


# 2c


In [8]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Example database
examples = [
    {"input": "The internet is down.",
     "output": "We are observing connectivity latency."},

    {"input": "This code implies a bug.",
     "output": "The logic suggests unintended behavior."},

    {"input": "I hate this feature.",
     "output": "This feature does not align with my preferences."},
]

# Format for ONE example
example_fmt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# Few-shot container
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_fmt,
    examples=examples
)

# Final prompt
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Corpo-Speak Translator. Rewrite the input professionally."),
    few_shot_prompt,
    ("human", "{text}")
])

chain = final_prompt | llm

print(chain.invoke({"text": "This app sucks."}).content)


The application's current iteration requires further optimization.
